# Space Weather: Terrestrial Perspective — 구현 / Implementation\n# Pulkkinen, T. (2007), Living Rev. Solar Phys., 4, 1\n\n---\n\n이 노트북은 Pulkkinen (2007) 리뷰 논문의 핵심 모델과 개념을 코드로 구현합니다.\n자기권계면 형태, 에너지 결합, Dst 예측, 전류계, 서브스톰, 복사대 역학, 전리권 가열, 그리고 지상 유도 전류(GIC)를 다룹니다.\n\nThis notebook implements the key models and concepts from Pulkkinen (2007).\nWe cover magnetopause shape, energy coupling, Dst prediction, current systems, substorms, radiation belt dynamics, ionospheric heating, and ground-induced currents (GIC).

In [ ]:
import numpy as np\nimport matplotlib.pyplot as plt\nfrom matplotlib.patches import Circle, FancyArrowPatch\nfrom matplotlib.collections import LineCollection\nimport matplotlib.gridspec as gridspec\nfrom scipy import constants\nfrom scipy.integrate import solve_ivp\nfrom scipy.special import erf\n\n# Physical constants from scipy.constants.\nR_E = 6.371e6  # Earth radius in meters.\nMU_0 = constants.mu_0  # Permeability of free space.\nE_CHARGE = constants.e  # Elementary charge.\nM_PROTON = constants.m_p  # Proton mass.\nM_ELECTRON = constants.m_e  # Electron mass.\nC_LIGHT = constants.c  # Speed of light.\nK_BOLTZ = constants.k  # Boltzmann constant.\n\n# Matplotlib style configuration.\nplt.rcParams.update({\n    'figure.figsize': (10, 7),\n    'font.size': 12,\n    'axes.grid': True,\n    'grid.alpha': 0.3,\n    'figure.dpi': 100,\n})

---\n## Part 1: 자기권계면 형태 — Shue 모델 / Magnetopause Shape — Shue Model\n\nShue et al. (1997, 1998)의 경험적 자기권계면 모델을 구현합니다 (논문 Eq 1-2).\n자기권계면은 태양풍 동압 $P_{sw}$와 IMF $B_z$의 함수로 기술됩니다:\n\nWe implement the empirical magnetopause model of Shue et al. (1997, 1998) (Paper Eq 1-2).\nThe magnetopause shape is described as a function of solar wind dynamic pressure $P_{sw}$ and IMF $B_z$:\n\n$$R(\\phi) = R_0 \\left(\\frac{2}{1 + \\cos\\phi}\\right)^\\alpha$$\n\n$$R_0 = (10.22 + 1.29 \\tanh[0.184(B_z + 8.14)]) \\cdot P_{sw}^{-1/6.6}$$\n\n$$\\alpha = (0.58 - 0.007 B_z)(1 + 0.024 \\ln P_{sw})$$\n\n여기서 $R_0$는 주간 기립 거리(subsolar standoff distance), $\\alpha$는 꼬리 플레어링 파라미터입니다.\n강한 자기 폭풍 시 $R_0$가 정지궤도(6.6 $R_E$) 안쪽으로 압축될 수 있습니다.\n\nHere $R_0$ is the subsolar standoff distance and $\\alpha$ is the tail flaring parameter.\nDuring strong storms, $R_0$ can compress inside geostationary orbit (6.6 $R_E$).

In [ ]:
def shue_magnetopause(phi: np.ndarray, bz: float, psw: float) -> np.ndarray:\n    """Compute magnetopause radial distance using Shue et al. (1998) model.\n\n    Args:\n        phi: Angle from the Earth-Sun line in radians (0 = subsolar point).\n        bz: IMF Bz component in nT.\n        psw: Solar wind dynamic pressure in nPa.\n\n    Returns:\n        Radial distance of the magnetopause in Earth radii.\n    """\n    r0 = (10.22 + 1.29 * np.tanh(0.184 * (bz + 8.14))) * psw ** (-1.0 / 6.6)\n    alpha = (0.58 - 0.007 * bz) * (1.0 + 0.024 * np.log(psw))\n    r = r0 * (2.0 / (1.0 + np.cos(phi))) ** alpha\n    return r\n\n\n# --- Define solar wind conditions. ---\nconditions = [\n    {"label": "Quiet (Bz=0, Psw=2 nPa)", "bz": 0, "psw": 2, "color": "royalblue", "ls": "-"},\n    {"label": "Moderate storm (Bz=−10, Psw=5 nPa)", "bz": -10, "psw": 5, "color": "orange", "ls": "--"},\n    {"label": "Extreme storm (Bz=−30, Psw=30 nPa)", "bz": -30, "psw": 30, "color": "red", "ls": "-."},\n]\n\nphi = np.linspace(-np.pi * 0.999, np.pi * 0.999, 500)\n\nfig, ax = plt.subplots(figsize=(10, 8))\n\nfor cond in conditions:\n    r = shue_magnetopause(phi, cond["bz"], cond["psw"])\n    # Convert to GSM X-Z plane (noon-midnight meridian).\n    x = r * np.cos(phi)\n    z = r * np.sin(phi)\n    r0_val = shue_magnetopause(np.array([0.0]), cond["bz"], cond["psw"])[0]\n    ax.plot(x, z, color=cond["color"], ls=cond["ls"], lw=2.0,\n            label=f'{cond["label"]} — R₀={r0_val:.1f} R_E')\n\n# Draw Earth.\nearth = Circle((0, 0), 1.0, color="steelblue", zorder=5)\nax.add_patch(earth)\nax.annotate("Earth", (0, 0), ha="center", va="center", fontsize=9,\n            fontweight="bold", color="white", zorder=6)\n\n# Draw geostationary orbit.\ntheta_geo = np.linspace(0, 2 * np.pi, 200)\nax.plot(6.6 * np.cos(theta_geo), 6.6 * np.sin(theta_geo),\n        'k:', lw=1.2, alpha=0.6, label="Geostationary orbit (6.6 R_E)")\n\nax.set_xlabel("X_GSM [R_E] (sunward →)")\nax.set_ylabel("Z_GSM [R_E]")\nax.set_title("Shue Magnetopause Model / Shue 자기권계면 모델\\n"\n             "(Noon-Midnight Meridian / 정오-자정 자오면)", fontsize=13)\nax.set_xlim(-30, 20)\nax.set_ylim(-35, 35)\nax.set_aspect("equal")\nax.axhline(0, color="gray", lw=0.5)\nax.axvline(0, color="gray", lw=0.5)\nax.legend(loc="lower left", fontsize=9)\nplt.tight_layout()\nplt.show()

---\n## Part 2: Akasofu Epsilon 파라미터 / Akasofu Epsilon Parameter\n\n태양풍에서 자기권으로의 에너지 전달률을 근사하는 경험적 파라미터입니다 (논문 Eq 3).\n$\\sin^4(\\theta/2)$ 의존성 때문에, 남향 IMF ($\\theta \\approx 180°$)일 때 에너지 전달이 극대화되고 북향 ($\\theta \\approx 0°$)에서는 거의 0입니다.\n\nThe empirical parameter approximating energy transfer rate from solar wind to magnetosphere (Paper Eq 3).\nDue to the $\\sin^4(\\theta/2)$ dependence, energy transfer maximizes for southward IMF ($\\theta \\approx 180°$) and nearly vanishes for northward ($\\theta \\approx 0°$).\n\n$$\\epsilon = 10^7 \\, V_{sw} \\, B^2 \\, (7 R_E)^2 \\, \\sin^4(\\theta/2)$$

In [ ]:
def akasofu_epsilon(v_sw: np.ndarray, b_imf: np.ndarray,\n                    theta: np.ndarray) -> np.ndarray:\n    """Compute the Akasofu epsilon parameter.\n\n    Args:\n        v_sw: Solar wind speed in m/s.\n        b_imf: IMF magnitude in Tesla.\n        theta: IMF clock angle in radians (0 = northward, pi = southward).\n\n    Returns:\n        Energy coupling parameter epsilon in Watts.\n    """\n    l0 = 7.0 * R_E  # Effective magnetopause cross-section radius.\n    return 1e7 * v_sw * b_imf**2 * l0**2 * np.sin(theta / 2.0)**4\n\n\n# --- (a) Epsilon vs clock angle for different B and V_sw. ---\ntheta_arr = np.linspace(0, np.pi, 200)\n\nfig, axes = plt.subplots(1, 2, figsize=(14, 5))\n\n# Panel (a): varying B at fixed V_sw.\nax = axes[0]\nv_fixed = 400e3  # 400 km/s.\nfor b_nt in [5, 10, 20]:\n    b_si = b_nt * 1e-9\n    eps = akasofu_epsilon(v_fixed, b_si, theta_arr)\n    ax.plot(np.degrees(theta_arr), eps / 1e12,\n            lw=2, label=f"B = {b_nt} nT")\nax.set_xlabel("IMF Clock Angle θ [°]")\nax.set_ylabel("ε [TW]")\nax.set_title("(a) ε vs θ (V_sw = 400 km/s) / ε 대 clock angle")\nax.legend()\nax.set_xlim(0, 180)\n\n# Panel (b): varying V_sw at fixed B.\nax = axes[1]\nb_fixed = 10e-9  # 10 nT.\nfor v_km in [300, 500, 800]:\n    v_si = v_km * 1e3\n    eps = akasofu_epsilon(v_si, b_fixed, theta_arr)\n    ax.plot(np.degrees(theta_arr), eps / 1e12,\n            lw=2, label=f"V_sw = {v_km} km/s")\nax.set_xlabel("IMF Clock Angle θ [°]")\nax.set_ylabel("ε [TW]")\nax.set_title("(b) ε vs θ (B = 10 nT) / ε 대 clock angle")\nax.legend()\nax.set_xlim(0, 180)\n\nplt.suptitle("Akasofu ε Parameter / Akasofu ε 파라미터", fontsize=14, y=1.02)\nplt.tight_layout()\nplt.show()

### 합성 태양풍과 ε 시계열 / Synthetic Solar Wind and ε Time Series\n\n합성 태양풍 데이터를 생성하고 ε 시계열을 계산합니다.\n간단한 $V_{sw} \\cdot B_s$ proxy와 비교하여, ε가 clock angle 의존성을 어떻게 포착하는지 보여줍니다.\n\nWe generate synthetic solar wind data and compute the ε time series.\nWe compare with the simple $V_{sw} \\cdot B_s$ proxy to show how ε captures the clock angle dependence.

In [ ]:
# --- Generate synthetic solar wind input over 24 hours. ---\nnp.random.seed(42)\nt_hours = np.linspace(0, 24, 1440)  # 1-minute resolution.\ndt_sec = (t_hours[1] - t_hours[0]) * 3600.0\n\n# Solar wind speed: baseline 400 km/s with a CME arrival at t=6h.\nv_sw_km = 400 + 200 * (1 / (1 + np.exp(-(t_hours - 6) * 3)))\n\n# IMF magnitude: increases during storm.\nb_imf_nt = 5 + 10 * np.exp(-0.5 * ((t_hours - 10) / 3)**2)\n\n# IMF clock angle: rotates from northward to southward and back.\ntheta_clock = np.pi * (0.2 + 0.8 * np.exp(-0.5 * ((t_hours - 12) / 4)**2))\n\n# Compute epsilon.\nv_sw_si = v_sw_km * 1e3\nb_imf_si = b_imf_nt * 1e-9\neps_ts = akasofu_epsilon(v_sw_si, b_imf_si, theta_clock)\n\n# Compute simple VBs proxy.\nbz = -b_imf_nt * np.cos(theta_clock)  # Bz from clock angle.\nbs = np.maximum(-bz, 0)  # Southward component only (positive when south).\nvbs = v_sw_km * bs * 1e-3  # mV/m units.\n\n# --- Plot. ---\nfig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)\n\naxes[0].plot(t_hours, v_sw_km, 'k', lw=1.5)\naxes[0].set_ylabel("V_sw [km/s]")\naxes[0].set_title("합성 태양풍 입력과 ε 시계열 / Synthetic Solar Wind Input & ε Time Series")\n\naxes[1].plot(t_hours, b_imf_nt, 'b', lw=1.5, label="|B|")\naxes[1].plot(t_hours, bz, 'r', lw=1.5, label="Bz")\naxes[1].set_ylabel("IMF [nT]")\naxes[1].legend(loc="upper right")\naxes[1].axhline(0, color="gray", lw=0.5)\n\naxes[2].plot(t_hours, eps_ts / 1e12, 'darkred', lw=1.5)\naxes[2].set_ylabel("ε [TW]")\naxes[2].set_ylim(bottom=0)\n\naxes[3].plot(t_hours, vbs, 'darkorange', lw=1.5, label="V·Bs proxy")\naxes[3].plot(t_hours, eps_ts / 1e12 * 0.5, 'darkred', lw=1.5, alpha=0.6,\n             label="ε (scaled)")\naxes[3].set_ylabel("V·Bs [mV/m] / ε (scaled)")\naxes[3].set_xlabel("Time [hours]")\naxes[3].legend()\n\nplt.tight_layout()\nplt.show()

---\n## Part 3: Burton/O'Brien-McPherron Dst 모델 / Burton/O'Brien-McPherron Dst Model\n\n개선된 Burton 공식 (논문 Eq 6-7, O'Brien & McPherron 2000)을 구현합니다.\n압력 보정된 $D_{st}^*$의 시간 진화를 태양풍 입력으로부터 예측하는 경험식입니다.\n\nWe implement the improved Burton formula (Paper Eq 6-7, O'Brien & McPherron 2000).\nThis empirical formula predicts the time evolution of pressure-corrected $D_{st}^*$ from solar wind input.\n\n$$D_{st}^* = D_{st} - 7.26\\sqrt{P_{sw}} + 11.0$$\n\n$$\\frac{dD_{st}^*}{dt} = Q(t) - \\frac{D_{st}^*}{\\tau}$$\n\n$$Q(t) = -4.4(V_{sw} B_s - E_c) \\quad \\text{for } V_{sw} B_s \\geq E_c, \\quad \\text{else } 0$$\n\n$$\\tau = 2.40 \\exp\\left(\\frac{9.74}{4.69 + V_{sw} B_s}\\right) \\quad \\text{[hours]}$$\n\n- $E_c = 0.49$ mV/m: 임계 전기장 / critical electric field\n- $\\tau$: 태양풍 구동에 의존하는 감쇠 시간 / decay time depending on solar wind driving

In [ ]:
def burton_dst_model(t_hours: np.ndarray, v_sw_km: np.ndarray,\n                     bz_nt: np.ndarray, psw_npa: np.ndarray,\n                     dst_star_init: float = 0.0) -> dict:\n    """Simulate Dst* evolution using the O'Brien-McPherron improved Burton model.\n\n    Args:\n        t_hours: Time array in hours.\n        v_sw_km: Solar wind speed in km/s.\n        bz_nt: IMF Bz in nT.\n        psw_npa: Solar wind dynamic pressure in nPa.\n        dst_star_init: Initial Dst* value in nT.\n\n    Returns:\n        Dictionary with Dst*, Dst, Q, and tau arrays.\n    """\n    ec = 0.49  # Critical electric field threshold [mV/m].\n    n = len(t_hours)\n    dst_star = np.zeros(n)\n    dst_star[0] = dst_star_init\n    q_arr = np.zeros(n)\n    tau_arr = np.zeros(n)\n\n    for i in range(n):\n        # Southward Bz component (positive when southward).\n        bs = max(-bz_nt[i], 0)  # nT\n        # V*Bs in mV/m: V [km/s] * Bs [nT] * 1e-3.\n        vbs = v_sw_km[i] * bs * 1e-3\n\n        # Injection function Q [nT/hr].\n        if vbs >= ec:\n            q_arr[i] = -4.4 * (vbs - ec)\n        else:\n            q_arr[i] = 0.0\n\n        # Decay timescale [hours].\n        tau_arr[i] = 2.40 * np.exp(9.74 / (4.69 + vbs))\n\n        # Euler integration.\n        if i < n - 1:\n            dt = t_hours[i + 1] - t_hours[i]\n            dst_star[i + 1] = dst_star[i] + dt * (q_arr[i] - dst_star[i] / tau_arr[i])\n\n    # Convert Dst* to observed Dst.\n    dst_obs = dst_star + 7.26 * np.sqrt(psw_npa) - 11.0\n\n    return {"dst_star": dst_star, "dst": dst_obs, "Q": q_arr, "tau": tau_arr}\n\n\n# --- Construct synthetic storm solar wind profile. ---\nt = np.linspace(0, 48, 2880)  # 48 hours, 1-min resolution.\n\n# Phase boundaries.\nt_ssc = 6.0    # Storm sudden commencement.\nt_main = 8.0   # Main phase onset.\nt_main_end = 14.0  # End of main phase.\n\n# Solar wind speed.\nv_profile = np.where(t < t_ssc, 400.0,\n            np.where(t < t_main, 400.0 + 100 * (t - t_ssc) / (t_main - t_ssc),\n            np.where(t < t_main_end, 500.0, 500.0 - 100 * (t - t_main_end) / 20)))\nv_profile = np.clip(v_profile, 400, 600)\n\n# IMF Bz.\nbz_profile = np.where(t < t_ssc, 5.0,\n             np.where(t < t_main, 5.0 - 25 * (t - t_ssc) / (t_main - t_ssc),\n             np.where(t < t_main_end, -20.0,\n             -20.0 + 25 * (t - t_main_end) / 10)))\nbz_profile = np.clip(bz_profile, -20, 5)\n\n# Dynamic pressure.\npsw_profile = np.where(t < t_ssc, 2.0,\n              np.where(t < t_ssc + 0.5, 2.0 + 13 * (t - t_ssc) / 0.5,\n              np.where(t < t_main, 15.0 - 10 * (t - t_ssc - 0.5) / (t_main - t_ssc - 0.5),\n              np.where(t < t_main_end, 5.0, 5.0 - 3 * (t - t_main_end) / 20))))\npsw_profile = np.clip(psw_profile, 2, 15)\n\n# Run the model.\nresult = burton_dst_model(t, v_profile, bz_profile, psw_profile)\n\n# --- Plot results. ---\nfig, axes = plt.subplots(4, 1, figsize=(13, 11), sharex=True)\n\naxes[0].plot(t, v_profile, 'k', lw=1.5)\naxes[0].set_ylabel("V_sw [km/s]")\naxes[0].set_title("Burton/O'Brien-McPherron Dst Model — 합성 자기 폭풍 / Synthetic Magnetic Storm")\n\naxes[1].plot(t, bz_profile, 'b', lw=1.5, label="Bz")\naxes[1].fill_between(t, bz_profile, 0, where=(bz_profile < 0), alpha=0.2, color='red')\naxes[1].axhline(0, color="gray", lw=0.5)\naxes[1].set_ylabel("IMF Bz [nT]")\naxes[1].legend()\n\nax2b = axes[1].twinx()\nax2b.plot(t, psw_profile, 'orange', lw=1.5, label="Psw")\nax2b.set_ylabel("Psw [nPa]", color="orange")\nax2b.legend(loc="upper right")\n\naxes[2].plot(t, result["dst_star"], 'darkred', lw=2, label="Dst*")\naxes[2].plot(t, result["dst"], 'b', lw=1.5, alpha=0.7, label="Dst (observed)")\naxes[2].axhline(0, color="gray", lw=0.5)\naxes[2].set_ylabel("Dst / Dst* [nT]")\naxes[2].legend()\n\n# Annotate storm phases.\nfor ax in axes:\n    ax.axvline(t_ssc, color='green', ls='--', alpha=0.5, lw=1)\n    ax.axvline(t_main, color='red', ls='--', alpha=0.5, lw=1)\n    ax.axvline(t_main_end, color='purple', ls='--', alpha=0.5, lw=1)\naxes[0].text(t_ssc + 0.1, 590, "SSC", color="green", fontsize=9)\naxes[0].text(t_main + 0.1, 590, "Main\\nphase", color="red", fontsize=8)\naxes[0].text(t_main_end + 0.1, 590, "Recovery", color="purple", fontsize=9)\n\naxes[3].plot(t, result["tau"], 'teal', lw=1.5)\naxes[3].set_ylabel("τ [hours]")\naxes[3].set_xlabel("Time [hours]")\naxes[3].set_title("감쇠 시간 τ / Decay Timescale τ")\n\nplt.tight_layout()\nplt.show()

### 파라미터 민감도 분석 / Parameter Sensitivity Analysis\n\nBz 크기와 지속 시간을 변화시켜 Dst* 반응의 민감도를 탐색합니다.\n\nWe explore the sensitivity of Dst* response by varying Bz magnitude and duration.

In [ ]:
# --- Sensitivity: vary Bz magnitude and main phase duration. ---\nfig, axes = plt.subplots(1, 2, figsize=(14, 5))\n\nt_sens = np.linspace(0, 48, 2880)\n\n# (a) Vary Bz magnitude, fixed 6-hour main phase.\nax = axes[0]\nfor bz_mag in [-5, -10, -20, -30]:\n    bz_s = np.where(t_sens < 6, 5.0,\n            np.where(t_sens < 12, float(bz_mag),\n            5.0))\n    v_s = np.full_like(t_sens, 450.0)\n    p_s = np.full_like(t_sens, 3.0)\n    res = burton_dst_model(t_sens, v_s, bz_s, p_s)\n    ax.plot(t_sens, res["dst_star"], lw=2, label=f"Bz = {bz_mag} nT")\nax.set_xlabel("Time [hours]")\nax.set_ylabel("Dst* [nT]")\nax.set_title("(a) Bz 크기 변화 / Vary Bz magnitude\\n(6-hr main phase, V=450 km/s)")\nax.legend(fontsize=9)\nax.axhline(0, color="gray", lw=0.5)\n\n# (b) Vary main phase duration, fixed Bz=-15 nT.\nax = axes[1]\nfor dur_hr in [2, 4, 6, 10]:\n    bz_s = np.where(t_sens < 6, 5.0,\n            np.where(t_sens < 6 + dur_hr, -15.0,\n            5.0))\n    v_s = np.full_like(t_sens, 450.0)\n    p_s = np.full_like(t_sens, 3.0)\n    res = burton_dst_model(t_sens, v_s, bz_s, p_s)\n    ax.plot(t_sens, res["dst_star"], lw=2, label=f"Duration = {dur_hr} hr")\nax.set_xlabel("Time [hours]")\nax.set_ylabel("Dst* [nT]")\nax.set_title("(b) 주상 지속 시간 변화 / Vary main phase duration\\n(Bz=−15 nT, V=450 km/s)")\nax.legend(fontsize=9)\nax.axhline(0, color="gray", lw=0.5)\n\nplt.tight_layout()\nplt.show()

---\n## Part 4: 자기권 전류계 / Magnetospheric Current Systems\n\n지구의 쌍극자 자기장과 주요 전류계를 시각화합니다.\n쌍극자장은 $B = B_0 (R_E/r)^3 \\sqrt{1 + 3\\sin^2\\lambda}$로 주어집니다.\nL-shell 구조와 입자 표류 경로(gradient-curvature drift + E x B drift)를 그리고,\n폭풍 시 강화된 대류 전기장이 고리 전류를 지구 쪽으로 밀어넣는 과정을 보여줍니다.\n\nWe visualize Earth's dipole magnetic field and major current systems.\nThe dipole field is $B = B_0 (R_E/r)^3 \\sqrt{1 + 3\\sin^2\\lambda}$.\nWe plot L-shell structure and particle drift paths (gradient-curvature drift + E x B drift),\nshowing how storm-enhanced convection electric field pushes the ring current earthward.

In [ ]:
def dipole_field_line(l_shell: float, n_points: int = 200) -> tuple:\n    """Compute a dipole magnetic field line for a given L-shell.\n\n    Args:\n        l_shell: L-shell parameter (equatorial crossing distance in R_E).\n        n_points: Number of points along the field line.\n\n    Returns:\n        Tuple of (x, z) coordinates in R_E.\n    """\n    theta = np.linspace(-np.pi / 2, np.pi / 2, n_points)\n    r = l_shell * np.cos(theta) ** 2\n    x = r * np.cos(theta)\n    z = r * np.sin(theta)\n    return x, z\n\n\ndef dipole_b_magnitude(r_re: np.ndarray, lam_rad: np.ndarray,\n                       b0: float = 3.12e-5) -> np.ndarray:\n    """Compute dipole magnetic field magnitude.\n\n    Args:\n        r_re: Radial distance in Earth radii.\n        lam_rad: Magnetic latitude in radians.\n        b0: Surface equatorial field strength in Tesla.\n\n    Returns:\n        Magnetic field magnitude in Tesla.\n    """\n    return b0 / r_re**3 * np.sqrt(1 + 3 * np.sin(lam_rad)**2)\n\n\n# --- Figure: Dipole field with current systems and L-shells. ---\nfig, axes = plt.subplots(1, 2, figsize=(16, 7))\n\n# --- Panel (a): Dipole field lines with schematic current systems. ---\nax = axes[0]\n\n# Draw field lines for various L-shells.\nfor L in [2, 3, 4, 5, 6, 8, 10]:\n    x, z = dipole_field_line(L)\n    mask = x**2 + z**2 >= 1.0  # Outside Earth.\n    ax.plot(x[mask], z[mask], 'b', lw=0.8, alpha=0.6)\n\n# Draw the magnetopause (Shue quiet).\nphi_mp = np.linspace(-np.pi * 0.85, np.pi * 0.85, 300)\nr_mp = shue_magnetopause(phi_mp, 0, 2)\nax.plot(r_mp * np.cos(phi_mp), r_mp * np.sin(phi_mp),\n        'k', lw=2.5, label="Magnetopause")\n\n# Earth.\nearth = Circle((0, 0), 1.0, color="steelblue", zorder=5)\nax.add_patch(earth)\n\n# Schematic current annotations.\nax.annotate("Ring\\nCurrent", xy=(4, 0), fontsize=9, color='red',\n            ha='center', fontweight='bold',\n            bbox=dict(boxstyle='round,pad=0.3', fc='lightyellow', alpha=0.8))\nax.annotate("Magnetopause\\nCurrent (Chapman-Ferraro)", xy=(10, 5), fontsize=8,\n            color='darkgreen', ha='center',\n            bbox=dict(boxstyle='round,pad=0.3', fc='lightgreen', alpha=0.7))\nax.annotate("Tail\\nCurrent", xy=(-8, 0), fontsize=9, color='purple',\n            ha='center', fontweight='bold',\n            bbox=dict(boxstyle='round,pad=0.3', fc='plum', alpha=0.7))\n\n# Ring current schematic (dashed arc at ~4 RE).\ntheta_rc = np.linspace(-np.pi * 0.8, np.pi * 0.8, 100)\nax.plot(4 * np.cos(theta_rc), 4 * np.sin(theta_rc), 'r--', lw=2, alpha=0.7)\n\n# Tail current sheet (horizontal line in nightside).\nax.plot([-12, -3], [0, 0], 'purple', lw=3, alpha=0.5)\n\nax.set_xlim(-15, 15)\nax.set_ylim(-12, 12)\nax.set_aspect('equal')\nax.set_xlabel('X [R_E] (sunward →)')\nax.set_ylabel('Z [R_E]')\nax.set_title('(a) 쌍극자장과 전류계 / Dipole Field & Current Systems')\nax.legend(loc='lower left', fontsize=8)\n\n# --- Panel (b): Particle drift paths (equatorial plane). ---\nax = axes[1]\n\n# Corotation + convection electric potential (Volland-Stern).\ndef volland_stern_potential(r_re: np.ndarray, phi: np.ndarray,\n                           kp: float = 2.0) -> np.ndarray:\n    """Compute Volland-Stern convection + corotation electric potential.\n\n    Args:\n        r_re: Radial distance in R_E (2D meshgrid).\n        phi: Azimuthal angle in radians (0 = noon, increasing dawnward).\n        kp: Kp index.\n\n    Returns:\n        Total electric potential in kV.\n    """\n    # Convection (Maynard-Chen parameterization).\n    a_conv = 0.045 / (1 - 0.159 * kp + 0.0093 * kp**2)**3\n    gamma = 2.0\n    phi_conv = a_conv * r_re**gamma * np.sin(phi)  # kV\n\n    # Corotation.\n    omega_e = 7.29e-5  # Earth rotation rate [rad/s].\n    b0 = 3.12e-5  # Surface equatorial B in T.\n    # Phi_corot = -omega * B0 * R_E^2 / r  (in SI, convert to kV)\n    phi_corot = -omega_e * b0 * (R_E**2) / (r_re * R_E) * 1e-3  # kV\n\n    return phi_conv + phi_corot\n\n\nr_arr = np.linspace(1.5, 10, 200)\nphi_arr = np.linspace(0, 2 * np.pi, 200)\nR, PHI = np.meshgrid(r_arr, phi_arr)\nX_eq = R * np.cos(PHI)\nY_eq = R * np.sin(PHI)\n\n# Quiet conditions.\npot_quiet = volland_stern_potential(R, PHI, kp=2)\n# Storm conditions.\npot_storm = volland_stern_potential(R, PHI, kp=6)\n\nearth2 = Circle((0, 0), 1.0, color='steelblue', zorder=5)\nax.add_patch(earth2)\n\n# Equipotentials = drift paths for cold plasma (E x B drift).\nlevels_q = np.linspace(-100, 20, 30)\ncs1 = ax.contour(X_eq, Y_eq, pot_quiet, levels=levels_q,\n                  colors='royalblue', linewidths=0.8, alpha=0.5)\ncs2 = ax.contour(X_eq, Y_eq, pot_storm, levels=levels_q,\n                  colors='red', linewidths=0.8, alpha=0.5, linestyles='--')\n\n# Manual legend.\nax.plot([], [], 'royalblue', lw=1.5, label='Quiet (Kp=2)')\nax.plot([], [], 'r--', lw=1.5, label='Storm (Kp=6)')\n\n# Geostationary orbit.\nax.plot(6.6 * np.cos(theta_geo), 6.6 * np.sin(theta_geo),\n        'k:', lw=1, alpha=0.5)\nax.annotate('GEO', xy=(6.6, 0.5), fontsize=8, color='gray')\n\nax.set_xlim(-10, 10)\nax.set_ylim(-10, 10)\nax.set_aspect('equal')\nax.set_xlabel('X [R_E] (sunward →)')\nax.set_ylabel('Y [R_E] (dawnward →)')\nax.set_title('(b) 입자 표류 경로 / Particle Drift Paths\\n(Equatorial Plane / 적도면)')\nax.legend(loc='lower left', fontsize=9)\nax.annotate('Sun →', xy=(8, -9), fontsize=10, color='orange', fontweight='bold')\n\nplt.tight_layout()\nplt.show()

---\n## Part 5: 서브스톰 주기 시뮬레이션 / Substorm Cycle Simulation\n\n간단한 자기 꼬리 자속 적재/방출 모델을 구현합니다:\n- **성장 단계 / Growth phase**: 개방 자속이 축적되고 lobe 자기장이 증가\n- **개시 / Onset**: 전류 시트가 임계 두께로 박화되면 재결합이 시작\n- **팽창 / Expansion**: 급격한 자속 폐합과 에너지 방출\n- **회복 / Recovery**: 정상 상태로 복귀\n\nHarris 전류 시트 모델 $B_x = B_0 \\tanh(Z/\\lambda)$도 구현하여 박화 과정을 보여줍니다.\n\nWe implement a simple magnetotail flux loading/unloading model:\n- **Growth phase**: open flux accumulates, lobe B increases\n- **Onset**: reconnection triggers when current sheet thins to threshold\n- **Expansion**: rapid flux closure, energy release\n- **Recovery**: return to quiet\n\nWe also implement the Harris current sheet model $B_x = B_0 \\tanh(Z/\\lambda)$ showing thinning.

In [ ]:
def simulate_substorm_cycle(t_min: np.ndarray,\n                            dayside_reconnection_rate: float = 0.08,\n                            nightside_reconnection_rate: float = 0.4,\n                            onset_threshold: float = 0.8,\n                            quiet_flux: float = 0.3) -> dict:\n    """Simulate a substorm cycle with flux loading and unloading.\n\n    Args:\n        t_min: Time array in minutes.\n        dayside_reconnection_rate: Open flux accumulation rate [Wb/s equivalent, normalized].\n        nightside_reconnection_rate: Flux closure rate during expansion [normalized].\n        onset_threshold: Normalized open flux threshold for substorm onset.\n        quiet_flux: Baseline open flux level.\n\n    Returns:\n        Dictionary with open_flux, lobe_b, cross_tail_j, al_proxy arrays.\n    """\n    n = len(t_min)\n    open_flux = np.zeros(n)\n    open_flux[0] = quiet_flux\n    phase = np.zeros(n, dtype=int)  # 0=quiet/growth, 1=expansion, 2=recovery.\n\n    for i in range(1, n):\n        dt = t_min[i] - t_min[i - 1]\n        if phase[i - 1] == 0:  # Growth phase: accumulate flux.\n            open_flux[i] = open_flux[i - 1] + dayside_reconnection_rate * dt / 60.0\n            if open_flux[i] >= onset_threshold:\n                phase[i] = 1  # Trigger onset.\n            else:\n                phase[i] = 0\n        elif phase[i - 1] == 1:  # Expansion: rapid flux closure.\n            open_flux[i] = open_flux[i - 1] - nightside_reconnection_rate * dt / 60.0\n            if open_flux[i] <= quiet_flux:\n                open_flux[i] = quiet_flux\n                phase[i] = 2\n            else:\n                phase[i] = 1\n        else:  # Recovery: slowly return to growth.\n            open_flux[i] = open_flux[i - 1] + 0.01 * dt / 60.0\n            if open_flux[i] >= quiet_flux + 0.05:\n                phase[i] = 0  # Resume growth.\n            else:\n                phase[i] = 2\n\n    # Derived quantities.\n    # Lobe magnetic pressure ~ B^2 ~ open_flux^2.\n    lobe_b = 20.0 * (open_flux / quiet_flux)  # nT, normalized.\n    # Cross-tail current density ~ d(open_flux)/dt during expansion.\n    cross_tail_j = np.gradient(open_flux, t_min / 60.0)  # Proxy.\n    # AL index proxy: spikes negative during expansion.\n    al_proxy = np.zeros(n)\n    al_proxy[phase == 1] = -500 * (open_flux[phase == 1] - quiet_flux) / (\n        onset_threshold - quiet_flux)\n    al_proxy[phase == 0] = -50 * (open_flux[phase == 0] - quiet_flux) / (\n        onset_threshold - quiet_flux)\n\n    return {\n        "open_flux": open_flux, "lobe_b": lobe_b,\n        "cross_tail_j": cross_tail_j, "al_proxy": al_proxy,\n        "phase": phase\n    }\n\n\n# Run three substorm cycles.\nt_sub = np.linspace(0, 360, 3600)  # 6 hours, 0.1 min resolution.\nresult_sub = simulate_substorm_cycle(t_sub)\n\n# --- Plot substorm cycle. ---\nfig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)\n\n# Color-code phases.\nphase_colors = {0: 'gold', 1: 'red', 2: 'lightblue'}\nphase_labels = {0: 'Growth', 1: 'Expansion', 2: 'Recovery'}\n\nfor ax in axes:\n    for p_val, p_color in phase_colors.items():\n        mask = result_sub[\"phase\"] == p_val\n        regions = np.where(np.diff(mask.astype(int)))[0]\n        starts = np.where(np.diff(mask.astype(int)) == 1)[0]\n        ends = np.where(np.diff(mask.astype(int)) == -1)[0]\n        if mask[0]:\n            starts = np.concatenate([[0], starts])\n        if mask[-1]:\n            ends = np.concatenate([ends, [len(mask) - 1]])\n        for s, e in zip(starts, ends):\n            ax.axvspan(t_sub[s], t_sub[min(e, len(t_sub) - 1)],\n                       alpha=0.15, color=p_color)\n\naxes[0].plot(t_sub, result_sub[\"open_flux\"], 'k', lw=1.5)\naxes[0].set_ylabel(\"Open Flux [normalized]\")\naxes[0].set_title(\"서브스톰 주기 시뮬레이션 / Substorm Cycle Simulation\")\naxes[0].axhline(0.8, color='red', ls=':', lw=1, alpha=0.7)\naxes[0].text(5, 0.82, \"Onset threshold\", fontsize=8, color='red')\n\naxes[1].plot(t_sub, result_sub[\"lobe_b\"], 'darkblue', lw=1.5)\naxes[1].set_ylabel(\"Lobe B [nT proxy]\")\n\naxes[2].plot(t_sub, result_sub[\"al_proxy\"], 'darkred', lw=1.5)\naxes[2].set_ylabel(\"AL Index [nT proxy]\")\naxes[2].set_xlabel(\"Time [minutes]\")\n\n# Phase legend.\nfor p_val, p_label in phase_labels.items():\n    axes[0].fill_between([], [], color=phase_colors[p_val], alpha=0.3,\n                          label=p_label)\naxes[0].legend(loc='upper right', fontsize=9)\n\nplt.tight_layout()\nplt.show()

### Harris 전류 시트 박화 / Harris Current Sheet Thinning\n\nHarris 전류 시트 모델: $B_x = B_0 \\tanh(Z/\\lambda)$.\n서브스톰 성장 단계에서 전류 시트가 $\\lambda = 2 R_E$에서 $\\lambda = 0.1 R_E$로 박화되는 과정을 보여줍니다.\n전류 밀도 $J_y \\propto \\text{sech}^2(Z/\\lambda)$는 박화 시 급격히 증가합니다.\n\nHarris current sheet model: $B_x = B_0 \\tanh(Z/\\lambda)$.\nDuring the substorm growth phase, the current sheet thins from $\\lambda = 2 R_E$ to $\\lambda = 0.1 R_E$.\nThe current density $J_y \\propto \\text{sech}^2(Z/\\lambda)$ increases dramatically during thinning.

In [ ]:
def harris_current_sheet(z_re: np.ndarray, b0: float,\n                         lam: float) -> tuple:\n    """Compute Harris current sheet magnetic field and current density.\n\n    Args:\n        z_re: Distance from neutral sheet in R_E.\n        b0: Lobe magnetic field strength in nT.\n        lam: Current sheet half-thickness in R_E.\n\n    Returns:\n        Tuple of (Bx in nT, Jy in nA/m^2 equivalent).\n    """\n    bx = b0 * np.tanh(z_re / lam)\n    # Current density: Jy = (B0 / (mu0 * lambda)) * sech^2(Z/lambda).\n    # Using normalized units for visualization.\n    jy = (b0 / lam) * (1.0 / np.cosh(z_re / lam))**2\n    return bx, jy\n\n\nz = np.linspace(-5, 5, 500)\nb0_lobe = 20.0  # nT\n\nlambdas = [2.0, 1.0, 0.5, 0.2, 0.1]\ncolors = plt.cm.hot_r(np.linspace(0.1, 0.8, len(lambdas)))\n\nfig, axes = plt.subplots(1, 2, figsize=(14, 6))\n\n# Panel (a): Bx profiles.\nax = axes[0]\nfor lam, c in zip(lambdas, colors):\n    bx, _ = harris_current_sheet(z, b0_lobe, lam)\n    ax.plot(z, bx, color=c, lw=2, label=f\"λ = {lam} R_E\")\nax.set_xlabel(\"Z [R_E]\")\nax.set_ylabel(\"B_x [nT]\")\nax.set_title(\"(a) Harris 전류 시트 자기장 / Harris Current Sheet B_x\")\nax.legend(fontsize=9)\nax.axhline(0, color='gray', lw=0.5)\nax.axvline(0, color='gray', lw=0.5)\n\n# Panel (b): Current density profiles.\nax = axes[1]\nfor lam, c in zip(lambdas, colors):\n    _, jy = harris_current_sheet(z, b0_lobe, lam)\n    ax.plot(z, jy, color=c, lw=2, label=f\"λ = {lam} R_E\")\nax.set_xlabel(\"Z [R_E]\")\nax.set_ylabel(\"J_y [nT/R_E, normalized]\")\nax.set_title(\"(b) 전류 밀도 / Current Density J_y\")\nax.legend(fontsize=9)\nax.set_xlim(-3, 3)\n\nplt.suptitle(\"Harris Current Sheet Thinning / Harris 전류 시트 박화\",\n             fontsize=14, y=1.02)\nplt.tight_layout()\nplt.show()

---\n## Part 6: 상대론적 전자 역학 / Relativistic Electron Dynamics\n\n외부 Van Allen 벨트의 상대론적 전자 역학을 다룹니다:\n- 전자 표류 주기 vs L-shell과 에너지\n- 방사 확산 방정식: $\\partial f/\\partial t = L^2 \\partial/\\partial L (D_{LL}/L^2 \\cdot \\partial f/\\partial L)$\n- ULF 파동 구동 확산에 의한 전자 증감\n- 폭풍 시 dropout-recovery 패턴\n- 플라즈마구 위치와 가속/손실 경계\n\nWe address relativistic electron dynamics in the outer Van Allen belt:\n- Electron drift period vs L-shell and energy\n- Radial diffusion equation: $\\partial f/\\partial t = L^2 \\partial/\\partial L (D_{LL}/L^2 \\cdot \\partial f/\\partial L)$\n- ULF wave-driven diffusion enhancing or depleting electrons\n- Storm-time dropout-recovery pattern\n- Plasmasphere location as boundary between acceleration and loss regions

In [ ]:
def electron_drift_period(l_shell: np.ndarray, energy_kev: float,\n                          b0: float = 3.12e-5) -> np.ndarray:\n    """Compute electron gradient-curvature drift period in a dipole field.\n\n    For equatorially mirroring electrons:\n        T_d = (pi * q * B0 * R_E^2) / (3 * L * E_k * (1 + E_k/(2*m_e*c^2)))\n    Simplified: T_d ~ 1.05 / (L * E_MeV) hours for relativistic electrons.\n\n    Args:\n        l_shell: L-shell values.\n        energy_kev: Electron kinetic energy in keV.\n\n    Returns:\n        Drift period in minutes.\n    """\n    energy_mev = energy_kev / 1000.0\n    # Approximate formula for equatorially mirroring particles.\n    t_drift_hours = 1.05 / (l_shell * energy_mev)  # hours.\n    return t_drift_hours * 60.0  # Convert to minutes.\n\n\n# --- (a) Drift period vs L-shell for various energies. ---\nfig, axes = plt.subplots(1, 2, figsize=(14, 6))\n\nl_arr = np.linspace(2, 8, 200)\nax = axes[0]\nfor e_kev in [100, 500, 1000, 3000, 5000]:\n    td = electron_drift_period(l_arr, e_kev)\n    ax.plot(l_arr, td, lw=2, label=f\"{e_kev} keV\")\nax.set_xlabel(\"L-shell\")\nax.set_ylabel(\"Drift Period [minutes]\")\nax.set_title(\"(a) 전자 표류 주기 / Electron Drift Period\")\nax.legend(fontsize=9)\nax.set_yscale('log')\nax.set_ylim(1, 1000)\n\n# --- (b) Radial diffusion simulation. ---\n# Solve: df/dt = L^2 d/dL (D_LL / L^2 * df/dL) - f/tau_loss\n# Using finite differences on a uniform L grid.\n\nnl = 100\nl_grid = np.linspace(2.0, 8.0, nl)\ndl = l_grid[1] - l_grid[0]\n\n# Diffusion coefficient: D_LL = D0 * L^10 (Schulz & Lanzerotti).\nd0 = 1e-10  # day^-1 RE^-2, normalized.\n\ndef radial_diffusion_step(f: np.ndarray, l: np.ndarray, dl: float,\n                          dt_day: float, d0: float,\n                          tau_loss: np.ndarray) -> np.ndarray:\n    """Advance radial diffusion equation by one time step (explicit Euler).\n\n    Args:\n        f: Phase space density array.\n        l: L-shell grid.\n        dl: Grid spacing.\n        dt_day: Time step in days.\n        d0: Diffusion coefficient normalization.\n        tau_loss: Loss timescale array in days.\n\n    Returns:\n        Updated phase space density.\n    """\n    f_new = f.copy()\n    for i in range(1, len(l) - 1):\n        d_ll_plus = d0 * ((l[i] + l[i + 1]) / 2)**10\n        d_ll_minus = d0 * ((l[i] + l[i - 1]) / 2)**10\n\n        # Flux: F = D_LL / L^2 * df/dL.\n        flux_plus = d_ll_plus / l[i]**2 * (f[i + 1] - f[i]) / dl\n        flux_minus = d_ll_minus / l[i]**2 * (f[i] - f[i - 1]) / dl\n\n        diffusion = l[i]**2 * (flux_plus - flux_minus) / dl\n        loss = -f[i] / tau_loss[i]\n\n        f_new[i] = f[i] + dt_day * (diffusion + loss)\n\n    f_new = np.maximum(f_new, 0)  # Ensure non-negative.\n    return f_new\n\n\n# Initial PSD: peaked at high L (source at outer boundary).\nf_init = np.exp(-0.5 * ((l_grid - 7.0) / 1.0)**2)\n\n# Loss timescale: short inside plasmasphere (L < 4), long outside.\nplasmapause_l = 4.0\ntau_loss = np.where(l_grid < plasmapause_l, 0.5, 10.0)  # Days.\n\n# Time integration.\ndt = 0.001  # Days.\nn_steps = 5000\nf = f_init.copy()\nsnapshots = {0: f_init.copy()}\nsnapshot_times = [0, 500, 1500, 3000, 5000]\n\nfor step in range(1, n_steps + 1):\n    f = radial_diffusion_step(f, l_grid, dl, dt, d0, tau_loss)\n    if step in snapshot_times:\n        snapshots[step] = f.copy()\n\n# Plot PSD evolution.\nax = axes[1]\ncolors_rd = plt.cm.viridis(np.linspace(0, 0.9, len(snapshot_times)))\nfor step, c in zip(snapshot_times, colors_rd):\n    time_days = step * dt\n    ax.plot(l_grid, snapshots[step], color=c, lw=2,\n            label=f\"t = {time_days:.1f} days\")\n\nax.axvline(plasmapause_l, color='cyan', ls='--', lw=1.5, alpha=0.7,\n           label=f\"Plasmapause (L={plasmapause_l})\")\nax.set_xlabel(\"L-shell\")\nax.set_ylabel(\"Phase Space Density f [normalized]\")\nax.set_title(\"(b) 방사 확산 시뮬레이션 / Radial Diffusion Simulation\")\nax.legend(fontsize=8)\n\nplt.suptitle(\"Relativistic Electron Dynamics / 상대론적 전자 역학\",\n             fontsize=14, y=1.02)\nplt.tight_layout()\nplt.show()

### 폭풍 시 전자 Dropout-Recovery 패턴 / Storm-time Dropout-Recovery Pattern\n\n자기 폭풍 주상 동안 외부 벨트 전자는 급격히 감소(dropout)하고, 회복 단계에서 ULF 파동에 의한 방사 확산으로 폭풍 전 수준 이상으로 증가(recovery)할 수 있습니다.\n플라즈마구 위치가 가속과 손실 영역의 경계 역할을 합니다.\n\nDuring the storm main phase, outer belt electrons drop rapidly, and during recovery, ULF wave-driven radial diffusion can enhance them above pre-storm levels.\nThe plasmasphere location acts as the boundary between acceleration and loss regions.

In [ ]:
# --- Simulate dropout-recovery pattern at L=5 during a storm. ---\nt_storm_days = np.linspace(0, 10, 1000)\n\n# Storm Dst profile.\ndst_storm = np.where(t_storm_days < 2, 0,\n            np.where(t_storm_days < 3, -100 * (t_storm_days - 2),\n            np.where(t_storm_days < 4, -100,\n            -100 * np.exp(-(t_storm_days - 4) / 3))))\n\n# Plasmasphere location: erodes during storm, recovers slowly.\npp_location = np.where(t_storm_days < 2, 5.0,\n              np.where(t_storm_days < 4, 5.0 - 2.0 * (t_storm_days - 2) / 2,\n              3.0 + 2.0 * (1 - np.exp(-(t_storm_days - 4) / 5))))\n\n# Relativistic electron flux at L=5 (schematic model).\n# Dropout during main phase, recovery with possible enhancement.\nflux_l5 = np.ones_like(t_storm_days) * 1e4  # Baseline cm^-2 s^-1 sr^-1.\n# Main phase dropout.\nmask_main = (t_storm_days >= 2.5) & (t_storm_days < 4)\nflux_l5[mask_main] = 1e4 * np.exp(-3 * (t_storm_days[mask_main] - 2.5))\n# Minimum.\nmask_min = (t_storm_days >= 4) & (t_storm_days < 5)\nflux_l5[mask_min] = 1e4 * np.exp(-3 * 1.5)\n# Recovery (enhanced above pre-storm).\nmask_rec = t_storm_days >= 5\nflux_l5[mask_rec] = 1e4 * np.exp(-3 * 1.5) + (\n    2e4 - 1e4 * np.exp(-3 * 1.5)) * (1 - np.exp(-(t_storm_days[mask_rec] - 5) / 2))\n\n# --- Plot. ---\nfig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)\n\naxes[0].plot(t_storm_days, dst_storm, 'k', lw=2)\naxes[0].set_ylabel(\"Dst [nT]\")\naxes[0].set_title(\"폭풍 시 전자 Dropout-Recovery / Storm Electron Dropout-Recovery\")\naxes[0].axhline(0, color='gray', lw=0.5)\n\naxes[1].semilogy(t_storm_days, flux_l5, 'darkred', lw=2)\naxes[1].set_ylabel(\"e⁻ Flux at L=5\\n[cm⁻² s⁻¹ sr⁻¹]\")\naxes[1].axhline(1e4, color='gray', ls=':', lw=1, alpha=0.5)\naxes[1].text(0.5, 1.2e4, \"Pre-storm level\", fontsize=9, color='gray')\naxes[1].annotate(\"Dropout\", xy=(3.5, 300), fontsize=10, color='red',\n                fontweight='bold')\naxes[1].annotate(\"Enhancement\", xy=(7, 1.5e4), fontsize=10, color='green',\n                fontweight='bold')\n\naxes[2].plot(t_storm_days, pp_location, 'cyan', lw=2, label=\"Plasmapause\")\naxes[2].axhline(5.0, color='gray', ls=':', lw=1, alpha=0.5)\naxes[2].set_ylabel(\"Plasmapause L\")\naxes[2].set_xlabel(\"Time [days]\")\naxes[2].legend()\naxes[2].set_ylim(2, 6)\n\n# Storm phase annotations.\nfor ax in axes:\n    ax.axvspan(2, 4, alpha=0.1, color='red')\n    ax.axvspan(4, 10, alpha=0.05, color='green')\naxes[0].text(2.5, 10, \"Main phase\", fontsize=9, color='red')\naxes[0].text(5, 10, \"Recovery phase\", fontsize=9, color='green')\n\nplt.tight_layout()\nplt.show()

---\n## Part 7: 전리권 Joule 가열과 입자 강수 / Ionospheric Joule Heating and Precipitation\n\n자기권에서 전리권으로 전달되는 에너지의 두 가지 주요 경로를 구현합니다 (논문 Eq 4-5):\n\nWe implement the two major energy dissipation pathways from magnetosphere to ionosphere (Paper Eq 4-5):\n\n**Joule 가열 / Joule Heating** (Eq 4):\n$$P_{JH} = 2 \\times 1.9 \\times 10^8 \\times AE \\quad [W]$$\n\n**입자 강수 / Particle Precipitation** (Eq 5):\n$$P_{PREC} = 2 \\times 10^9 \\times (4.4\\sqrt{|AL|} - 7.6) \\quad [W]$$\n\n또한 자기권 에너지 입력의 분배(고리 전류, Joule 가열, 입자 강수, 꼬리 plasmoid)를 파이 차트로 보여줍니다.\n\nWe also show the energy budget pie chart: how total magnetospheric input is partitioned (ring current, Joule heating, precipitation, tail plasmoid).

In [ ]:
def joule_heating_power(ae_nt: np.ndarray) -> np.ndarray:\n    """Compute ionospheric Joule heating power (both hemispheres).\n\n    Args:\n        ae_nt: AE index in nT.\n\n    Returns:\n        Joule heating power in Watts.\n    """\n    return 2.0 * 1.9e8 * ae_nt\n\n\ndef precipitation_power(al_nt: np.ndarray) -> np.ndarray:\n    """Compute particle precipitation power (both hemispheres).\n\n    Args:\n        al_nt: AL index magnitude (positive value) in nT.\n\n    Returns:\n        Precipitation power in Watts.\n    """\n    return 2.0e9 * (4.4 * np.sqrt(al_nt) - 7.6)\n\n\n# --- Panel (a): Power vs AE/AL indices. ---\nfig = plt.figure(figsize=(15, 6))\ngs = gridspec.GridSpec(1, 3, width_ratios=[1, 1, 0.8])\n\n# Joule heating vs AE.\nax1 = fig.add_subplot(gs[0])\nae_arr = np.linspace(0, 1500, 300)\np_jh = joule_heating_power(ae_arr)\nax1.plot(ae_arr, p_jh / 1e12, 'darkorange', lw=2.5)\nax1.set_xlabel(\"AE Index [nT]\")\nax1.set_ylabel(\"Joule Heating Power [TW]\")\nax1.set_title(\"(a) Joule 가열 / Joule Heating (Eq 4)\")\nax1.fill_between(ae_arr, 0, p_jh / 1e12, alpha=0.15, color='orange')\n\n# Precipitation vs |AL|.\nax2 = fig.add_subplot(gs[1])\nal_arr = np.linspace(10, 1500, 300)  # Avoid sqrt(0) region.\np_prec = precipitation_power(al_arr)\np_prec = np.maximum(p_prec, 0)  # Physically non-negative.\nax2.plot(al_arr, p_prec / 1e12, 'darkgreen', lw=2.5)\nax2.set_xlabel(\"|AL| Index [nT]\")\nax2.set_ylabel(\"Precipitation Power [TW]\")\nax2.set_title(\"(b) 입자 강수 / Particle Precipitation (Eq 5)\")\nax2.fill_between(al_arr, 0, p_prec / 1e12, alpha=0.15, color='green')\n\n# --- Panel (c): Energy budget pie chart. ---\n# Approximate energy partitioning during a moderate storm\n# (based on Pulkkinen 2007 discussion of energy budget).\nax3 = fig.add_subplot(gs[2])\n\nlabels = [\n    'Joule Heating\\n(Joule 가열)\\n~35%',\n    'Ring Current\\n(고리 전류)\\n~20%',\n    'Particle Precip.\\n(입자 강수)\\n~15%',\n    'Tail Plasmoid\\n(꼬리 plasmoid)\\n~15%',\n    'Other\\n(기타)\\n~15%'\n]\nsizes = [35, 20, 15, 15, 15]\ncolors_pie = ['#FF9800', '#F44336', '#4CAF50', '#9C27B0', '#78909C']\nexplode = (0.05, 0, 0, 0, 0)\n\nax3.pie(sizes, explode=explode, labels=labels, colors=colors_pie,\n        autopct='', startangle=90, textprops={'fontsize': 8})\nax3.set_title(\"(c) 에너지 분배 / Energy Budget\", fontsize=11)\n\nplt.suptitle(\"Ionospheric Energy Dissipation / 전리권 에너지 소산\",\n             fontsize=14, y=1.02)\nplt.tight_layout()\nplt.show()

---\n## Part 8: 우주 기상 효과 — GIC와 실용적 영향 / Space Weather Effects — GIC and Practical Impacts\n\n급격한 지자기장 변화가 지상 도체(전력망, 파이프라인)에 유도하는 전류(GIC)를 모델링합니다.\n서브스톰 onset과 SSC(Storm Sudden Commencement) 시의 $dB/dt$ 스파이크가 GIC를 유발하는 과정을 보여주고,\nCME에서 지상 효과까지의 완전한 우주 기상 이벤트 타임라인을 제시합니다.\n마지막으로 Schwenn (태양 관점)과 Pulkkinen (지구 관점)의 비교 요약표를 작성합니다.\n\nWe model geomagnetically induced currents (GIC) driven by rapid geomagnetic field variations.\nWe show how dB/dt spikes during substorm onset and SSC create GIC spikes,\npresent a complete timeline from CME to ground effects,\nand provide a summary comparison table between Schwenn (solar) and Pulkkinen (terrestrial) perspectives.

In [ ]:
def simple_gic_model(db_dt: np.ndarray, ground_resistivity: float = 1e-3,\n                     conductor_length: float = 100e3,\n                     conductor_resistance: float = 0.1) -> dict:\n    """Compute GIC from dB/dt using a simple 1D model.\n\n    Faraday's law: E = -dB/dt * depth_scale (simplified).\n    The induced electric field drives current in a grounded conductor.\n\n    Args:\n        db_dt: Time derivative of magnetic field in nT/s.\n        ground_resistivity: Ground resistivity in Ohm*m.\n        conductor_length: Length of grounded conductor in meters.\n        conductor_resistance: Total resistance of conductor in Ohms.\n\n    Returns:\n        Dictionary with E_field (V/km) and GIC (Amperes).\n    """\n    # Skin depth approximation for induced E-field.\n    # E ~ dB/dt * sqrt(rho / (mu0 * omega)), simplified to E ~ dB/dt * scale.\n    scale_factor = np.sqrt(ground_resistivity / MU_0)  # Approximate.\n    e_field = np.abs(db_dt) * 1e-9 * scale_factor  # V/m.\n    e_field_per_km = e_field * 1e3  # V/km.\n\n    # GIC = E * L / R.\n    gic = e_field * conductor_length / conductor_resistance\n\n    return {\"e_field_vkm\": e_field_per_km, \"gic_amps\": gic}\n\n\n# --- Synthetic geomagnetic field with SSC and substorm onset. ---\nt_min_gic = np.linspace(0, 120, 7200)  # 2 hours, 1s resolution.\ndt_min = t_min_gic[1] - t_min_gic[0]\n\n# Construct B_H (horizontal component) with SSC and substorm.\nb_h = np.zeros_like(t_min_gic)\n\n# Quiet baseline.\nb_h += 25000  # nT (typical mid-latitude H).\n\n# SSC at t=20 min: sharp positive pulse.\nssc_mask = (t_min_gic >= 20) & (t_min_gic < 22)\nb_h[ssc_mask] += 50 * np.sin(np.pi * (t_min_gic[ssc_mask] - 20) / 2)\n\n# Gradual decrease (main phase beginning).\nmain_mask = t_min_gic >= 22\nb_h[main_mask] -= 30 * (1 - np.exp(-(t_min_gic[main_mask] - 22) / 30))\n\n# Substorm onset at t=60 min: sharp negative bay.\nsub_center = 60.0\nsub_width = 1.5  # Minutes (very rapid).\nb_h -= 200 * np.exp(-0.5 * ((t_min_gic - sub_center) / sub_width)**2)\n\n# Add some noise.\nnp.random.seed(123)\nb_h += np.random.normal(0, 2, len(t_min_gic))\n\n# Compute dB/dt.\ndb_dt = np.gradient(b_h, dt_min * 60.0)  # nT/s.\n\n# Compute GIC.\ngic_result = simple_gic_model(db_dt)\n\n# --- Plot. ---\nfig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)\n\naxes[0].plot(t_min_gic, b_h, 'k', lw=1.2)\naxes[0].set_ylabel(\"B_H [nT]\")\naxes[0].set_title(\"GIC 모델: SSC + 서브스톰 / GIC Model: SSC + Substorm Onset\")\naxes[0].annotate(\"SSC\", xy=(20, b_h[np.argmin(np.abs(t_min_gic - 20))]),\n                xytext=(25, 25060), fontsize=10, color='green',\n                arrowprops=dict(arrowstyle='->', color='green'))\naxes[0].annotate(\"Substorm\\nOnset\", xy=(60, b_h[np.argmin(np.abs(t_min_gic - 60))]),\n                xytext=(70, 24850), fontsize=10, color='red',\n                arrowprops=dict(arrowstyle='->', color='red'))\n\naxes[1].plot(t_min_gic, db_dt, 'darkblue', lw=1.2)\naxes[1].set_ylabel(\"dB/dt [nT/s]\")\naxes[1].axhline(0, color='gray', lw=0.5)\n\naxes[2].plot(t_min_gic, gic_result[\"gic_amps\"], 'darkred', lw=1.2)\naxes[2].set_ylabel(\"GIC [A]\")\naxes[2].set_xlabel(\"Time [minutes]\")\naxes[2].set_ylim(bottom=0)\n\nplt.tight_layout()\nplt.show()

### 우주 기상 이벤트 타임라인 / Space Weather Event Timeline\n\nCME 발생에서 지상 효과까지의 완전한 우주 기상 이벤트 시퀀스를 시각화합니다.\n\nWe visualize the complete sequence of a space weather event from CME eruption to ground effects.

In [ ]:
# --- Space weather event timeline visualization. ---\nfig, ax = plt.subplots(figsize=(14, 8))\nax.set_xlim(-1, 11)\nax.set_ylim(-0.5, 10.5)\nax.axis('off')\n\n# Timeline events: (time_label, event_kr, event_en, y_pos, color).\nevents = [\n    (\"T = 0 h\", \"CME 발생\", \"CME Eruption at Sun\", 10, \"#FF5722\"),\n    (\"T ~ 8 min\", \"X선 플레어 관측\", \"X-ray Flare Observed\", 9, \"#FF9800\"),\n    (\"T ~ 10-60 min\", \"SEP 도달 (양성자)\", \"SEP Arrival (protons)\", 8, \"#FFC107\"),\n    (\"T ~ 18-80 h\", \"ICME 지구 도달\", \"ICME Arrival at Earth\", 7, \"#4CAF50\"),\n    (\"T + 0 min\", \"SSC (자기권 압축)\", \"SSC (Magnetosphere Compression)\", 6, \"#2196F3\"),\n    (\"T + 0.5-2 h\", \"주상 시작 (Bz 남향)\", \"Main Phase Onset (Bz Southward)\", 5, \"#3F51B5\"),\n    (\"T + 1-6 h\", \"서브스톰 & 고리 전류 강화\", \"Substorms & Ring Current Enhancement\", 4, \"#9C27B0\"),\n    (\"T + 1-6 h\", \"전리권 폭풍 & GPS 오차\", \"Ionospheric Storm & GPS Errors\", 3, \"#E91E63\"),\n    (\"T + 1-6 h\", \"GIC 스파이크 (전력망 위협)\", \"GIC Spikes (Power Grid Threat)\", 2, \"#F44336\"),\n    (\"T + 1-10 d\", \"회복 단계 & 전자 증강\", \"Recovery Phase & Electron Enhancement\", 1, \"#607D8B\"),\n]\n\nfor time_label, event_kr, event_en, y, color in events:\n    # Timeline marker.\n    ax.plot(1.5, y, 'o', color=color, markersize=12, zorder=5)\n    ax.plot([1.5, 2.5], [y, y], '-', color=color, lw=2)\n\n    # Time label.\n    ax.text(0.3, y, time_label, fontsize=9, va='center',\n            fontweight='bold', color='#333')\n\n    # Event descriptions.\n    ax.text(2.7, y + 0.15, event_en, fontsize=10, va='center', color=color,\n            fontweight='bold')\n    ax.text(2.7, y - 0.15, event_kr, fontsize=9, va='center', color='#555')\n\n# Vertical timeline bar.\nax.plot([1.5, 1.5], [0.5, 10.5], '-', color='gray', lw=1.5, alpha=0.5)\n\n# Domain labels.\nax.text(9, 10, \"Solar\\n태양\", fontsize=10, color='#FF5722',\n        ha='center', fontweight='bold',\n        bbox=dict(boxstyle='round', fc='#FFF3E0', alpha=0.8))\nax.text(9, 7, \"Heliosphere\\n태양권\", fontsize=10, color='#4CAF50',\n        ha='center', fontweight='bold',\n        bbox=dict(boxstyle='round', fc='#E8F5E9', alpha=0.8))\nax.text(9, 4, \"Magnetosphere\\n자기권\", fontsize=10, color='#3F51B5',\n        ha='center', fontweight='bold',\n        bbox=dict(boxstyle='round', fc='#E8EAF6', alpha=0.8))\nax.text(9, 1.5, \"Ground\\n지상\", fontsize=10, color='#F44336',\n        ha='center', fontweight='bold',\n        bbox=dict(boxstyle='round', fc='#FFEBEE', alpha=0.8))\n\n# Domain boundary lines.\nfor y_line in [6.5, 3.5, 0.5]:\n    ax.axhline(y_line, color='gray', ls=':', lw=0.8, alpha=0.4)\n\nax.set_title(\"우주 기상 이벤트 타임라인 / Space Weather Event Timeline\\n\"\n             \"(CME → Ground Effects)\", fontsize=14, pad=20)\n\nplt.tight_layout()\nplt.show()

### Schwenn vs Pulkkinen 비교 요약 / Comparison Summary\n\n이 두 LRSP 리뷰는 우주 기상을 상보적인 관점에서 다룹니다.\n\nThese two LRSP reviews cover space weather from complementary perspectives.\n\n| 측면 / Aspect | Schwenn (2006) — 태양 관점 / Solar | Pulkkinen (2007) — 지구 관점 / Terrestrial |\n|---|---|---|\n| **초점 / Focus** | 태양 활동, CME, 플레어, SEP, 태양풍 / Solar activity, CME, flares, SEP, solar wind | 자기권, 전리권, 고리 전류, 복사대 / Magnetosphere, ionosphere, ring current, radiation belt |\n| **핵심 물리 / Core Physics** | 코로나 자기장, 재결합, 충격파 / Coronal B-field, reconnection, shocks | Dungey cycle, 서브스톰, 입자 표류 / Dungey cycle, substorms, particle drifts |\n| **주요 관측 / Key Observations** | SOHO/LASCO 코로나그래프, GOES X선 / SOHO/LASCO coronagraph, GOES X-ray | ACE/WIND L1 모니터, IMAGE, DMSP / ACE/WIND L1 monitor, IMAGE, DMSP |\n| **시뮬레이션 / Simulations** | 경험적 CME 전파 모델 / Empirical CME propagation | 전역 MHD (GUMICS-4, LFM) / Global MHD |\n| **예보 도구 / Forecasting** | CME 도착 시간 경험식 / CME arrival time empirical formula | Burton/Dst 예측, AI 모델 / Burton/Dst prediction, AI models |\n| **시간 범위 / Timescale** | 태양 표면 → L1 (수일) / Solar surface → L1 (days) | 자기권계면 → 지상 (수분-수시간) / Magnetopause → ground (min-hours) |\n| **실용적 영향 / Practical Impacts** | SEP 방사선 위험, 통신 두절 / SEP radiation hazard, comm blackout | 위성 대전, GPS 오차, GIC, 전력망 / Satellite charging, GPS error, GIC, power grid |\n| **핵심 파라미터 / Key Parameters** | CME 속도, X선 등급, 양성자 플럭스 / CME speed, X-ray class, proton flux | ε, Dst, Kp, AE, dB/dt / ε, Dst, Kp, AE, dB/dt |

---\n## 요약 / Summary\n\n이 노트북에서 구현한 내용:\n\nWhat we implemented in this notebook:\n\n| Part | 내용 / Content | 핵심 수식/모델 / Key Equation/Model |\n|------|---------------|------------------------------------|\n| 1 | 자기권계면 형태 / Magnetopause Shape | Shue et al. (1998): $R(\\phi) = R_0 (2/(1+\\cos\\phi))^\\alpha$ |\n| 2 | Akasofu ε 파라미터 / Epsilon Parameter | $\\epsilon = 10^7 V_{sw} B^2 (7R_E)^2 \\sin^4(\\theta/2)$ |\n| 3 | Burton/O'Brien-McPherron Dst 모델 | $dD_{st}^*/dt = Q(t) - D_{st}^*/\\tau$ |\n| 4 | 자기권 전류계 / Current Systems | Dipole field + Volland-Stern convection |\n| 5 | 서브스톰 주기 / Substorm Cycle | Flux loading/unloading + Harris current sheet |\n| 6 | 상대론적 전자 / Relativistic Electrons | Radial diffusion: $\\partial f/\\partial t = L^2 \\partial/\\partial L (D_{LL}/L^2 \\cdot \\partial f/\\partial L)$ |\n| 7 | 전리권 에너지 소산 / Ionospheric Dissipation | Joule heating (Eq 4) + Precipitation (Eq 5) |\n| 8 | GIC와 실용적 영향 / GIC & Practical Impacts | $dB/dt$ → E-field → GIC in conductor |\n\n---\n\n## References\n\n- Pulkkinen, T., \"Space Weather: Terrestrial Perspective\", Living Rev. Solar Phys., 4, 1 (2007). DOI: 10.12942/lrsp-2007-1\n- Shue, J.-H. et al., \"Magnetopause location under extreme solar wind conditions\", J. Geophys. Res., 103, 17691 (1998).\n- Burton, R. K. et al., \"An empirical relationship between interplanetary conditions and Dst\", J. Geophys. Res., 80, 4204 (1975).\n- O'Brien, T. P. & McPherron, R. L., \"An empirical phase space analysis of ring current dynamics\", J. Geophys. Res., 105, 7707 (2000).\n- Akasofu, S.-I., \"Energy coupling between the solar wind and the magnetosphere\", Space Sci. Rev., 28, 121 (1981).\n- Schulz, M. & Lanzerotti, L. J., \"Particle Diffusion in the Radiation Belts\", Springer (1974).\n- Schwenn, R., \"Space Weather: The Solar Perspective\", Living Rev. Solar Phys., 3, 2 (2006).